In [19]:
# Instalar gensim
!pip install -q gensim

import gensim
print("Gensim:", gensim.__version__)

Gensim: 4.4.0


In [20]:
# Bloque 0b. Montar Drive y verificar que el CSV ha subido entero
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/TFM_FreshMind/'
CSV_DRIVE = RUTA + 'RecipeNLG_dataset.csv'
!ls -lh "{CSV_DRIVE}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 2.2G Dec  9  2020 /content/drive/MyDrive/TFM_FreshMind/RecipeNLG_dataset.csv


In [21]:
# Bloque 1. Muestra aleatoria y reproducible leyendo por trozos.
import pandas as pd, numpy as np, time

CSV = RUTA + 'RecipeNLG_dataset.csv'
COLS = ['title', 'ingredients', 'directions', 'NER']
FRACCION = 0.09

rng = np.random.default_rng(42)
trozos, inicio = [], time.time()

for i, trozo in enumerate(pd.read_csv(CSV, usecols=COLS, chunksize=200000)):
    trozos.append(trozo.sample(frac=FRACCION, random_state=int(rng.integers(1e6))))
    print(f"  trozo {i+1} leído")

recetas = pd.concat(trozos, ignore_index=True)

print(f"\nMuestra real: {len(recetas)} recetas en {time.time()-inicio:.0f} s")
print(recetas.head(2))

  trozo 1 leído
  trozo 2 leído
  trozo 3 leído
  trozo 4 leído
  trozo 5 leído
  trozo 6 leído
  trozo 7 leído
  trozo 8 leído
  trozo 9 leído
  trozo 10 leído
  trozo 11 leído
  trozo 12 leído

Muestra real: 200803 recetas en 72 s
            title                                        ingredients  \
0  Angel Biscuits  ["1 pkg. active dry yeast", "5 c. all-purpose ...   
1   Arroz Con Buo  ["1 lb. chicken breast, skinned", "1 pkg. Goya...   

                                          directions  \
0  ["Dissolve yeast in warm water.", "Sift all dr...   
1  ["First, you take the corn oil.", "Place it in...   

                                                 NER  
0  ["active dry yeast", "flour", "baking powder",...  
1  ["chicken breast", "Sazon", "tomato sauce", "c...  


In [22]:
# Bloque 2. Convertir la columna NER en listas limpias de ingredientes.
# RecipeNLG ya trae en NER los ingredientes extraídos sin cantidades.
import ast
from collections import Counter

def limpiar_ner(texto):
    """'["Milk", "Olive Oil"]' -> ['milk', 'olive oil']"""
    try:
        lista = ast.literal_eval(texto)
    except Exception:
        return []
    return [i.strip().lower() for i in lista if isinstance(i, str) and i.strip()]

recetas['ingredientes'] = recetas['NER'].apply(limpiar_ner)
recetas['n_ing'] = recetas['ingredientes'].apply(len)

# Fuera recetas con muy pocos ingredientes (ruido) o demasiados (atípicas)
antes = len(recetas)
recetas = recetas[(recetas['n_ing'] >= 3) & (recetas['n_ing'] <= 20)].reset_index(drop=True)
print(f"Recetas antes: {antes} -> tras filtrar: {len(recetas)}")

# Vocabulario del corpus
conteo = Counter(i for lista in recetas['ingredientes'] for i in lista)
print("Ingredientes distintos:", len(conteo))
print("Los 15 más frecuentes:", conteo.most_common(15))

# Guardamos la muestra limpia
recetas.to_parquet(RUTA + 'recetas_200k_limpias.parquet', index=False)
print("Guardado en Drive.")

Recetas antes: 200803 -> tras filtrar: 195013
Ingredientes distintos: 40960
Los 15 más frecuentes: [('salt', 88436), ('sugar', 57915), ('butter', 47615), ('flour', 42928), ('eggs', 37341), ('onion', 34634), ('garlic', 33081), ('milk', 32868), ('water', 30812), ('vanilla', 25699), ('olive oil', 19496), ('pepper', 17004), ('brown sugar', 16420), ('egg', 15530), ('tomatoes', 15092)]
Guardado en Drive.


In [23]:
# Bloque 3. Word2Vec: cada ingrediente se convierte en un vector de 100 números.
# Los ingredientes que aparecen juntos en muchas recetas quedan cerca en el
# espacio vectorial.
from gensim.models import Word2Vec
import time

frases = recetas['ingredientes'].tolist()

inicio = time.time()
w2v = Word2Vec(
    sentences=frases,
    vector_size=100,
    window=10,
    min_count=5,       # ignora ingredientes con menos de 5 apariciones
    sg=1,              # skip-gram: mejor con vocabulario grande y disperso
    epochs=5,
    workers=4,
    seed=42
)
print(f"Entrenado en {time.time()-inicio:.0f} s")
print(f"Vocabulario útil: {len(w2v.wv)} ingredientes (de {len(conteo)} distintos)")

Entrenado en 31 s
Vocabulario útil: 7205 ingredientes (de 40960 distintos)


In [24]:
# Bloque 3b. Validación cualitativa: lo que ha aprendido tiene sentido?
for ing in ['chicken', 'olives', 'yogurt', 'salmon', 'spinach', 'cottage cheese']:
    if ing in w2v.wv:
        vecinos = [v for v, _ in w2v.wv.most_similar(ing, topn=5)]
        print(f"{ing:16} -> {vecinos}")
    else:
        print(f"{ing:16} -> NO está en el vocabulario")

chicken          -> ['chicken breasts', 'chicken breast', 'chicken meat', 'rotisserie chicken', 'boneless chicken']
olives           -> ['black olives', 'ripe olives', 'chopped ripe olives', 'green olives', 'thin slices hard salami']
yogurt           -> ['plain yogurt', 'greek yogurt', 'low-fat plain yogurt', 'low-fat yogurt', 'nonfat yogurt']
salmon           -> ['salmon fillet', 'fillet', 'trout', 'salmon fillets', 'tilapia']
spinach          -> ['fresh spinach', 'frozen spinach', 'chopped spinach', 'fresh spinach leaves', 'baby spinach']
cottage cheese   -> ['cream-style cottage cheese', 'ricotta cheese', 'low-fat cottage cheese', 'manicotti shells', 'whole basil']


In [25]:
# Bloque 4. Puente FoodKeeper (español) -> vocabulario Word2Vec (inglés).
# La usuaria elige productos en español; el motor busca en inglés. Usamos la
# traducción oficial del USDA (cruce por ID), no traducción automática.
import pandas as pd, re

prod_en = pd.read_excel(RUTA + 'FoodKeeper-Data.xls',    sheet_name='Product')
prod_es = pd.read_excel(RUTA + 'FoodKeeper-Data-ES.xls', sheet_name='Product')

mapa = prod_en[['ID', 'Name', 'Name_subtitle']].copy()
mapa['name_en'] = (mapa['Name'].fillna('') + ' ' +
                   mapa['Name_subtitle'].fillna('')).str.strip()

# Nombre español construido igual que en el notebook 01, para que encaje
es = prod_es[['ID', 'Name', 'Name_subtitle']].copy()
es['nombre_es'] = (es['Name'].fillna('') +
    es['Name_subtitle'].fillna('').apply(lambda s: ' - ' + s if s else ''))
mapa = mapa.merge(es[['ID', 'nombre_es']], on='ID', how='left')

def buscar_token(texto):
    """Busca el término del vocabulario que mejor representa el producto.
    Prueba todas las combinaciones de palabras, de la más larga a la más corta:
    'Cheese - Parmesan, grated' probará 'cheese parmesan' antes que 'cheese'."""
    limpio = re.sub(r'[^a-z ]', ' ', str(texto).lower())
    pal = limpio.split()
    cands = [' '.join(pal[i:j]) for i in range(len(pal)) for j in range(i+1, len(pal)+1)]
    cands.sort(key=lambda c: -len(c.split()))
    for c in cands:
        if c in w2v.wv:
            return c
    return None

mapa['token'] = mapa['name_en'].apply(buscar_token)

print(f"Productos con token: {mapa['token'].notna().sum()} de {len(mapa)}")
print(mapa[['nombre_es', 'name_en', 'token']].dropna(subset=['token']).head(12).to_string())

Productos con token: 633 de 661
                                                         nombre_es                                             name_en           token
0                                                      Mantequilla                                              Butter          butter
1                                Suero de mantequilla (Buttermilk)                                          Buttermilk      buttermilk
2   Queso - Quesos duros como cheddar, suizo, parmesano en bloque   Cheese hard such as cheddar, swiss, block parmesan          cheese
3                                      Queso - Parmesano, rallado                  Cheese parmesan; shredded or grated          cheese
4           Queso - Rayado,  queso cheddar, queso mozzarella, etc.          Cheese shredded; cheddar, mozzarella, etc.          cheese
5                                Queso - Queso procesado, rebanado                             Cheese processed slices          cheese
6               Queso -

In [26]:
# Bloque 4c. Mejora del puente: preferir el token especifico al genérico.
# Problema detectado: 5 tipos de queso distintos apuntaban todos a 'cheese'.
# Entonces recorremos los candidatos de mayor a menor número de palabras y
# nos quedamos con el primero que exista, descartando los de 1 sola palabra
# hasta haber agotado los de 2 y 3.
import re

def buscar_token_v2(texto):
    limpio = re.sub(r'[^a-z ]', ' ', str(texto).lower())
    pal = limpio.split()
    cands = []
    for n in (3, 2, 1):
        for i in range(len(pal) - n + 1):
            cands.append(' '.join(pal[i:i+n]))
    for c in cands:
        if c in w2v.wv:
            return c
    return None

mapa['token'] = mapa['name_en'].apply(buscar_token_v2)

ok = mapa.dropna(subset=['token'])
print(f"Cobertura: {len(ok)}/{len(mapa)} = {len(ok)/len(mapa)*100:.1f}%")
print(f"Tokens distintos: {ok['token'].nunique()}  (antes eran menos)\n")
print(ok[['nombre_es', 'name_en', 'token']].head(12).to_string())

Cobertura: 633/661 = 95.8%
Tokens distintos: 423  (antes eran menos)

                                                         nombre_es                                             name_en           token
0                                                      Mantequilla                                              Butter          butter
1                                Suero de mantequilla (Buttermilk)                                          Buttermilk      buttermilk
2   Queso - Quesos duros como cheddar, suizo, parmesano en bloque   Cheese hard such as cheddar, swiss, block parmesan          cheese
3                                      Queso - Parmesano, rallado                  Cheese parmesan; shredded or grated          cheese
4           Queso - Rayado,  queso cheddar, queso mozzarella, etc.          Cheese shredded; cheddar, mozzarella, etc.          cheese
5                                Queso - Queso procesado, rebanado                             Cheese processed slices  

In [27]:
# Bloque 4d. Diagnóstico: qué productos comparten el mismo token?
peores = ok['token'].value_counts().head(10)
print("Tokens que absorben más productos:")
print(peores.to_string())

print("\nEjemplo del peor caso:")
t = peores.index[0]
print(ok[ok['token'] == t][['nombre_es', 'name_en']].head(8).to_string())

Tokens que absorben más productos:
token
ham               19
lamb               8
fish               8
fruit              7
cream              7
vegetables         5
cheese             5
milk               5
salad dressing     5
pork               5

Ejemplo del peor caso:
                                                          nombre_es                                            name_en
76             Jamón - Enlatados (etiqueta  "Mantenga Refrigerado")             Ham canned (“keep refrigerated” label)
77                                Jamón - Cocido, con hueso, entero                   Ham fully cooked, bone-in, whole
78                                 Jamón - Cocido, con hueso, mitad                    Ham fully cooked, bone-in, half
79                        Jamón - Cocido, rebanado en forma espiral      Ham fully cooked, slices, half, or spiral cut
80  Jamón - Cocido, carne del cuarto delantero "Picnic", deshuesado    Ham fully cooked, arm picnic shoulder, boneless
81         

In [28]:
# Bloque 4e. Corrección del orden invertido de catálogo.
# El FoodKeeper escribe "Cheese parmesan" (orden de catálogo) pero el corpus
# de recetas usa "parmesan cheese" (orden natural del inglés). Probamos
# ambas direcciones en los pares de palabras.
def buscar_token_v3(texto):
    limpio = re.sub(r'[^a-z ]', ' ', str(texto).lower())
    pal = limpio.split()
    cands = []
    for i in range(len(pal) - 2):
        cands.append(' '.join(pal[i:i+3]))
    for i in range(len(pal) - 1):
        cands.append(f"{pal[i]} {pal[i+1]}")
        cands.append(f"{pal[i+1]} {pal[i]}")
    cands += pal
    for c in cands:
        if c in w2v.wv:
            return c
    return None

mapa['token'] = mapa['name_en'].apply(buscar_token_v3)
ok = mapa.dropna(subset=['token'])

print(f"Cobertura: {len(ok)}/{len(mapa)} = {len(ok)/len(mapa)*100:.1f}%")
print(f"Tokens distintos: {ok['token'].nunique()}  (con v2 eran 423)\n")

# Comprobamos justo los casos que fallaban
print(ok[ok['name_en'].str.contains('Cheese|Cream', case=False, na=False)]
      [['name_en', 'token']].head(10).to_string())

Cobertura: 633/661 = 95.8%
Tokens distintos: 456  (con v2 eran 423)

                                               name_en             token
2   Cheese hard such as cheddar, swiss, block parmesan       hard cheese
3                  Cheese parmesan; shredded or grated   parmesan cheese
4           Cheese shredded; cheddar, mozzarella, etc.   shredded cheese
5                              Cheese processed slices  processed cheese
6            Cheese soft such as brie, bel paese, goat       soft cheese
7                   Coffee creamer liquid refrigerated    coffee creamer
8                                       Cottage cheese    cottage cheese
9                                         Cream cheese      cream cheese
10                    Cream whipping, ultrapasteurized    whipping cream
11                            Cream whipped, sweetened     whipped cream


In [29]:
# Bloque 4f. Limpieza adicional del corpus.
# La inspección de resultados reveló dos problemas en la columna NER:
# (1) ingredientes duplicados dentro de una misma receta, que distorsionan
# el vector medio; (2) ruido del extractor ('your', 'south', verbos).
# Filtramos por frecuencia: un ingrediente real aparece en muchas recetas.
MIN_FREC = 50
validos = {ing for ing, n in conteo.items() if n >= MIN_FREC}
print(f"Ingredientes que superan el filtro: {len(validos)} de {len(conteo)}")

def depurar(lista):
    """Quita ruido y duplicados, conservando el orden original."""
    visto, salida = set(), []
    for i in lista:
        if i in validos and i not in visto:
            visto.add(i)
            salida.append(i)
    return salida

recetas['ingredientes'] = recetas['ingredientes'].apply(depurar)
recetas['n_ing'] = recetas['ingredientes'].apply(len)

antes = len(recetas)
recetas = recetas[recetas['n_ing'] >= 3].reset_index(drop=True)
print(f"Recetas: {antes} -> {len(recetas)} tras exigir 3 ingredientes válidos")
print("\nEjemplo depurado:", recetas['ingredientes'].iloc[0])

Ingredientes que superan el filtro: 1688 de 40960
Recetas: 195013 -> 190095 tras exigir 3 ingredientes válidos

Ejemplo depurado: ['active dry yeast', 'flour', 'baking powder', 'salt', 'buttermilk', 'soda', 'sugar', 'vegetable shortening']


In [30]:
# Bloque 5a. Vector de cada receta = media de los vectores de sus ingredientes
import numpy as np, time

DIM = 100
inicio = time.time()

def vector_medio(ingredientes, pesos=None):
    """Media (ponderada) de los vectores de los ingredientes conocidos."""
    vecs, ws = [], []
    for k, ing in enumerate(ingredientes):
        if ing in w2v.wv:
            vecs.append(w2v.wv[ing])
            ws.append(1 if pesos is None else pesos[k])
    if not vecs:
        return None
    return np.average(vecs, axis=0, weights=ws)

matriz = np.zeros((len(recetas), DIM), dtype='float32')
for i, lista in enumerate(recetas['ingredientes']):
    v = vector_medio(lista)
    if v is not None:
        matriz[i] = v

normas = np.linalg.norm(matriz, axis=1, keepdims=True)
normas[normas == 0] = 1
matriz_norm = matriz / normas

print(f"Matriz de recetas: {matriz_norm.shape} en {time.time()-inicio:.0f} s")

Matriz de recetas: (190095, 100) en 16 s


### Iteración: detección de falta de diversidad

La primera versión del recomendador ordenaba las recetas solo por similitud
coseno y número de ingredientes urgentes. La prueba reveló un problema:

    [1 urgentes | sim 0.90] Crunchy and Sweet Munch Mix
    [1 urgentes | sim 0.89] Pretzel Turtles
    [1 urgentes | sim 0.87] Party Mix
    [1 urgentes | sim 0.87] Smackin' Good Snack Mix
    [1 urgentes | sim 0.86] Cracker Snack Mix

Las cinco sugerencias son variantes del mismo plato. Es un comportamiento
conocido del vector medio con similitud coseno: las recetas casi duplicadas
ocupan posiciones contiguas del ranking. Una aplicación que ofrece cinco
veces la misma receta no resuelve el problema del usuario, así que se añadió
un filtro de diversidad basado en el índice de Jaccard (bloque siguiente).

In [31]:
# Bloque 5b. Recomendador con filtro de diversidad.
# Problema detectado: las 5 recetas devueltas eran casi idénticas entre sí.
# Solución: selección voraz descartando recetas cuyo solape de ingredientes
# (índice de Jaccard) con una ya elegida supere el umbral.
PESO_URGENCIA = {'rojo': 3, 'naranja': 2, 'verde': 1}
# Usamos 'ok' (productos CON token) y no 'mapa', para que el diccionario
# no contenga valores nulos
token_de = dict(zip(ok['nombre_es'], ok['token']))

def recomendar(inventario, top=5, max_solape=0.5):
    ings, pesos = [], []
    for nombre, urg in inventario:
        t = token_de.get(nombre)
        if t:
            ings.append(t); pesos.append(PESO_URGENCIA[urg])
    if not ings:
        return "Ningún producto del inventario se pudo enlazar"

    v = vector_medio(ings, pesos)
    sim = matriz_norm @ (v / np.linalg.norm(v))
    urgentes = {token_de.get(n) for n, u in inventario if u == 'rojo'}

    res = recetas[['title', 'ingredientes']].copy()
    res['similitud'] = sim
    res['usa_urgentes'] = res['ingredientes'].apply(lambda l: len(urgentes & set(l)))
    # Preseleccionamos 200 y luego diversificamos
    res = res.sort_values(['usa_urgentes', 'similitud'], ascending=False).head(200)

    elegidas = []
    for _, r in res.iterrows():
        s = set(r['ingredientes'])
        repetida = any(len(s & set(e['ingredientes'])) / len(s | set(e['ingredientes']))
                       >= max_solape for e in elegidas)
        if not repetida:
            elegidas.append(r)
        if len(elegidas) == top:
            break
    return pd.DataFrame(elegidas)

In [32]:
# Bloque 5d. Prueba final con un inventario realista.
# Nota: el FoodKeeper no cataloga la espinaca como producto independiente;
# la agrupa en "Bolsa de Vegetales" (token 'greens'). Se documenta como
# limitación del catálogo de origen en la sección 6.
def producto_por_token(tok):
    hit = ok[ok['token'] == tok]
    return hit.iloc[0]['nombre_es'] if len(hit) else None

deseados = [('chicken', 'rojo'), ('greens', 'rojo'),
            ('yogurt', 'naranja'), ('white rice', 'verde')]

inventario = []
for tok, urg in deseados:
    nombre = producto_por_token(tok)
    if nombre:
        inventario.append((nombre, urg))

print("Inventario simulado:")
for n, u in inventario:
    print(f"  [{u:8}] {n}  ->  {token_de[n]}")

print("\nRecetas recomendadas:")
for _, r in recomendar(inventario).iterrows():
    print(f"  [{r['usa_urgentes']} urgentes | sim {r['similitud']:.2f}] {r['title']}")
    print(f"      {r['ingredientes'][:8]}")

Inventario simulado:
  [rojo    ] Pollo - Entero  ->  chicken
  [rojo    ] Verduras  ->  greens
  [naranja ] Yogur  ->  yogurt
  [verde   ] Arroz - Blanco o mixto  ->  white rice

Recetas recomendadas:
  [1 urgentes | sim 0.88] Curried Chicken & Lentil Salad - 2.5-Qt Pressure Cooker
      ['vegetable oil', 'chicken', 'dried lentils', 'water', 'curry powder', 'grapes', 'cashews', 'celery']
  [1 urgentes | sim 0.88] Tandoori Chicken Salad
      ['chicken', 'butternut squash', 'baby spinach', 'red onion', 'plain yogurt', 'lemon juice', 'cilantro', 'tahini']
  [1 urgentes | sim 0.87] Chicken And Mango Salad
      ['chicken', 'baby spinach', 'rotisserie chicken', 'mangoes', 'red onion', 'cashews', 'white wine vinegar', 'olive oil']
  [1 urgentes | sim 0.87] Crazy Spicy Chicken
      ['tomatoes', 'mango', 'vidalia onion', 'serrano peppers', 'white pepper', 'yellow mustard', 'curry powder', 'salt']
  [1 urgentes | sim 0.87] Lemon Chia Greek Yogurt Chicken and Zucchini Noodles
      ['yogurt',

In [33]:
# Bloque 6. Guardar artefactos ligeros
# Decisión de productivización: no llevamos gensim a producción. Solo
# necesita los vectores de los 456 tokens del FoodKeeper y la matriz de
# recetas, así que guardamos numpy puro
import os

N_APP = 25000        # submuestra
idx = np.random.default_rng(42).choice(len(recetas), N_APP, replace=False)

recetas_app = recetas.iloc[idx][['title', 'ingredientes', 'directions']].reset_index(drop=True)
recetas_app.to_parquet(RUTA + 'recetas_app.parquet', index=False)
np.save(RUTA + 'matriz_app.npy', matriz_norm[idx].astype('float32'))

# Solo los vectores que se necesitan
tokens_app = sorted(ok['token'].unique())
vectores = np.vstack([w2v.wv[t] for t in tokens_app]).astype('float32')
np.save(RUTA + 'vectores_tokens.npy', vectores)

# Diccionario producto español -> token, para los desplegables
ok[['nombre_es', 'token']].to_parquet(RUTA + 'mapa_productos.parquet', index=False)

for f in ['recetas_app.parquet', 'matriz_app.npy',
          'vectores_tokens.npy', 'mapa_productos.parquet']:
    print(f"{f:25} {os.path.getsize(RUTA + f)/1e6:6.1f} MB")

recetas_app.parquet          7.3 MB
matriz_app.npy              10.0 MB
vectores_tokens.npy          0.2 MB
mapa_productos.parquet       0.0 MB


In [34]:
# Bloque 7. Artefactos definitivos
# Unimos el puente (nombre_es -> token) con la categoría que usa el modelo
# del Módulo 1, y guardamos el vocabulario en un fichero explícito para que
# no dependa del orden en que se generaron los vectores.
features = pd.read_parquet(RUTA + 'features_modulo1.parquet')
cat = features[['nombre_es', 'categoria_es']].drop_duplicates('nombre_es')

productos = ok[['nombre_es', 'token']].merge(cat, on='nombre_es', how='left')
print("Sin categoría asignada:", productos['categoria_es'].isna().sum())
productos = productos.dropna(subset=['categoria_es']).drop_duplicates('nombre_es')
productos.to_parquet(RUTA + 'productos_app.parquet', index=False)

# Vocabulario y vectores en el mismo orden, guardados juntos
vocab = sorted(productos['token'].unique())
vectores = np.vstack([w2v.wv[t] for t in vocab]).astype('float32')
np.save(RUTA + 'vectores_tokens.npy', vectores)
pd.DataFrame({'token': vocab}).to_parquet(RUTA + 'vocab_app.parquet', index=False)

print(f"Productos para la app: {len(productos)} | tokens: {len(vocab)}")

Sin categoría asignada: 11
Productos para la app: 619 | tokens: 449


In [35]:
# Bloque 8. requirements.txt con las versiones exactas de este entorno.
# Evita que Streamlit Cloud instale versiones distintas y falle al abrir
# los .pkl generados aquí.
import sklearn, xgboost, joblib

req = (f"streamlit\npandas=={pd.__version__}\nnumpy=={np.__version__}\n"
       f"scikit-learn=={sklearn.__version__}\nxgboost=={xgboost.__version__}\n"
       f"joblib=={joblib.__version__}\npyarrow\n")
with open(RUTA + 'requirements.txt', 'w') as f:
    f.write(req)
print(req)

streamlit
pandas==2.2.3
numpy==2.1.3
scikit-learn==1.6.1
xgboost==3.4.1
joblib==1.6.0
pyarrow



In [36]:
# Bloque 9. Combinaciones categoría-estado-lugar presentes en los datos.
# La app solo ofrecerá estas: evita que el modelo prediga sobre casos que
# nunca vio (p. ej. un congelado guardado en la despensa), donde la
# extrapolación produce resultados poco fiables.
combos = (features[['categoria_es', 'estado', 'lugar']]
          .drop_duplicates().reset_index(drop=True))
combos.to_parquet(RUTA + 'combos_validos.parquet', index=False)

print(f"Combinaciones válidas: {len(combos)}")
print("\nLo que el modelo SÍ ha visto para 'Congelados':")
print(combos[combos['categoria_es'] == 'Congelados'].to_string(index=False))

Combinaciones válidas: 83

Lo que el modelo SÍ ha visto para 'Congelados':
categoria_es       estado      lugar
  Congelados      cerrado     nevera
  Congelados      cerrado congelador
  Congelados descongelado     nevera
